In [0]:
%pip install duckdb
import duckdb 
import pandas as pd 
import numpy as np
import os

# Orange Cap Dataset

In [0]:
df_orange_cap = pd.read_csv("/Workspace/Users/manikanthgoud98@gmail.com/ipl_data_modelling/data/raw/ipl_orange_cap.csv")
df_orange_cap

# Purple Cap Dataset

In [0]:
df_purple_cap = pd.read_csv("/Workspace/Users/manikanthgoud98@gmail.com/ipl_data_modelling/data/raw/ipl_purple_cap.csv")
df_purple_cap

In [0]:
# The case=False makes it ignore capital letters
# The na=False prevents errors if there are empty rows
starc_data = df_purple_cap[df_purple_cap['player'] == 'mitchell starc']
starc_data

# 2022 Auction Dataset

In [0]:
df_2022_auction = pd.read_csv("/Workspace/Users/manikanthgoud98@gmail.com/ipl_data_modelling/data/raw/IPL_2022_Sold_Players.csv")
df_2022_auction["Auction Year"] = 2022
df_2022_auction.rename(columns={'Price Paid':'Sold (Cr)', 'Players':'player'}, inplace=True)
if df_2022_auction['Sold (Cr)'].dtype == object:
    df_2022_auction['Sold (Cr)'] = df_2022_auction['Sold (Cr)'].str.replace('₹', '').str.replace(',', '').astype(float) / 10000000
else:
    df_2022_auction['Sold (Cr)'] = df_2022_auction['Sold (Cr)'] / 10000000
df_2022_auction

# 2023 Auction Dataset

In [0]:
df_2023_auction = pd.read_csv("/Workspace/Users/manikanthgoud98@gmail.com/ipl_data_modelling/data/raw/IPL_2023_Auction_Sold.csv")
df_2023_auction['player'] = df_2023_auction['First Name'] + ' ' + df_2023_auction['Surname']
df_2023_auction['Auction Year'] = 2023
df_2023_auction.rename(columns={'Auction_Price':'Sold (Cr)','Specialism':'Type','Country':'Nationality'}, inplace=True)
df_2023_auction['Nationality'] = np.where(df_2023_auction['Nationality'] == 'India', 'Indian', 'Overseas')
df_2023_auction["Type"].replace({'BATSMAN': 'Batsman', 'BOWLER': 'Bowler', 'ALL-ROUNDER': 'All-Rounder', 'WICKETKEEPER': 'Wicket Keeper'},inplace=True)
df_2023_auction["Sold (Cr)"] = df_2023_auction["Sold (Cr)"] / 100

df_2023_auction

In [0]:
df_2023_auction["Type"].unique()

In [0]:
df_2023_auction_cleaned = df_2023_auction[["player", "Nationality", "Type", "Sold (Cr)", "TEAM", "Auction Year"]]
df_2023_auction_cleaned

# 2024 Auction Dataset

In [0]:
df_2024_auction = pd.read_csv("/Workspace/Users/manikanthgoud98@gmail.com/ipl_data_modelling/data/raw/2024_auction.csv")

# Modify the dataframe in place directly
df_2024_auction.rename(columns={'PRICE PAID': 'Sold (Cr)', 'PLAYER':'player','TYPE':'Type', 'NATIONALITY':'Nationality','TEAM':'Team'}, inplace=True)
df_2024_auction['Team'].replace({'Kolkata Knight Riders': 'KKR', 'Royal Challengers Bengaluru': 'RCB', 'Mumbai Indians': 'MI', 'Chennai Super Kings': 'CSK', 'Rajasthan Royals': 'RR', 'Delhi Capitals': 'DC', 'Sunrisers Hyderabad': 'SRH', 'Gujarat Titans': 'GT', 'Lucknow Super Giants': 'LSG', 'Punjab Kings': 'PBKS'}, inplace=True)
df_2024_auction["Sold (Cr)"] = df_2024_auction["Sold (Cr)"] / 10000000
df_2024_auction['Auction Year'] = 2024

df_2024_auction

In [0]:
df_2024_auction['Team'].unique()

# 2025 Auction Dataset

In [0]:
df_2025_auction = pd.read_csv("/Workspace/Users/manikanthgoud98@gmail.com/ipl_data_modelling/data/raw/ipl_2025_auction_players.csv")
df_2025_auction.rename(columns={'Sold': 'Sold (Cr)', 'Players':'player'}, inplace=True)
df_2025_auction['Type'].replace({'BAT': 'Batsman', 'BOWL': 'Bowler', 'AR': 'All-Rounder', 'WK': 'Wicket Keeper'}, inplace=True)
df_2025_auction['Auction Year'] = 2025
df_2025_auction_cleaned = df_2025_auction[['player', 'Team', 'Type', 'Sold (Cr)', 'Auction Year']]


In [0]:
df_2025_auction['Type'].unique()

# Combine all auctions into a single dataframe

In [0]:
auction_cols = ['player', 'Team', 'Type', 'Sold (Cr)', 'Auction Year']

df_auction = pd.concat([
    df_2022_auction.reindex(columns=auction_cols),
    df_2023_auction_cleaned.reindex(columns=auction_cols),
    df_2024_auction.reindex(columns=auction_cols),
    df_2025_auction_cleaned.reindex(columns=auction_cols)
], ignore_index=True)

df_auction

## Cleaning Names

In [0]:
import re


def clean_name(name):
    if pd.isna(name):
        return name
    name = str(name).lower()         # 1. Convert to lowercase
    name = name.strip()              # 2. Remove leading/trailing spaces
    name = name.replace('.', '')     # 3. Remove periods (e.g., M.S. -> MS)
    name = re.sub(' +', ' ', name)   # 4. Replace double spaces with single space
    return name

In [0]:
df_auction["player"] = df_auction["player"].apply(clean_name)
df_auction

In [0]:
df_orange_cap["player"] = df_orange_cap["player"].apply(clean_name)
df_orange_cap


In [0]:
df_purple_cap["player"] = df_purple_cap["player"].apply(clean_name)
df_purple_cap

# Merge Orange Cap and Purple Cap

In [0]:
df_perf = pd.merge(
    df_orange_cap, 
    df_purple_cap, 
    on=['player', 'year'], 
    how='outer', 
    suffixes=('_bat', '_bowl')
)

In [0]:
df_perf

# Extra cleaning of the merged datasets

In [0]:
auction_names = set(df_auction['player'].unique())
perf_names = set(df_perf['player'].unique())

# Find players in the auction that DO NOT exist in the performance data
unmatched_in_auction = auction_names - perf_names

print(f"There are {len(unmatched_in_auction)} unmatched names in the auction data.")
print(list(unmatched_in_auction)[:20]) # Print first 20 to inspect

In [0]:
def create_join_key(name):
    if pd.isna(name):
        return None
    name = str(name).lower()
    # Remove EVERYTHING that isn't a lowercase letter (removes spaces, dots, hyphens)
    name = re.sub(r'[^a-z]', '', name)
    return name

In [0]:
# ---------------------------------------------------------
# STEP 1: CLEAN AUCTION DATA BEFORE MERGING
# ---------------------------------------------------------
# Drop the "ghost" rows where the player name is completely missing
df_auction = df_auction.dropna(subset=['player'])

# Apply the hack to create a matching key
df_auction['join_key'] = df_auction['player'].apply(create_join_key)

In [0]:
df_auction

In [0]:
# Drop ghost rows here too
df_perf = df_perf.dropna(subset=['player'])

# Apply the exact same hack
df_perf['join_key'] = df_perf['player'].apply(create_join_key)

In [0]:
df_perf

In [0]:
df_perf['Target_Auction_Year'] = df_perf['year'] + 1

In [0]:
df_perf

# Merging Auction and Performance

In [0]:
final_records = []

for index, auction_row in df_auction.iterrows():
    player_key = auction_row['join_key']
    auction_year = auction_row['Auction Year']
    
    # Find past performances before the auction
    past_performances = df_perf[(df_perf['join_key'] == player_key) & (df_perf['year'] < auction_year)]
    
    if past_performances.empty:
        combined_data = auction_row.to_dict()
        combined_data['Is_Debutant'] = 1
    else:
        most_recent_season = past_performances.sort_values('year', ascending=False).iloc[0]
        combined_data = {**auction_row.to_dict(), **most_recent_season.to_dict()}
        combined_data['Is_Debutant'] = 0
        
    final_records.append(combined_data)

df_final = pd.DataFrame(final_records)

In [0]:
df_final

In [0]:
df_final['Is_Debutant'] = df_final['matches_bat'].isnull() & df_final['matches_bowl'].isnull()
df_final['Is_Debutant'] = df_final['Is_Debutant'].astype(int)
df_final

In [0]:
df_final = df_final.fillna(0)
df_final

In [0]:
# # 1. Clean up column names
# df_final = df_final.rename(columns={'player_x': 'Player'})
# df_final = df_final.drop(columns=['player_y', 'year', 'Team'], errors='ignore') 
# # (Note: Dropping 'Team' for the model because you usually want to predict a player's intrinsic value, not what a specific team will pay)

# # 2. Fix the "Best Bowling" String
# # Split "5/24" into Best_Wkts = 5 and Best_Runs = 24
# df_final[['best_bowl_wkts', 'best_bowl_runs']] = df_final['best_bowling'].str.split('/', expand=True)

# # Convert to numeric, filling errors/NaNs with 0
# df_final['best_bowl_wkts'] = pd.to_numeric(df_final['best_bowl_wkts'], errors='coerce').fillna(0)
# df_final['best_bowl_runs'] = pd.to_numeric(df_final['best_bowl_runs'], errors='coerce').fillna(0)

# # Drop the original string column
# df_final = df_final.drop(columns=['best_bowling'])

In [0]:
df_final["Sold (Cr)"].unique()

In [0]:
df_final.info()

In [0]:
df_final['Sold (Cr)'] = pd.to_numeric(df_final['Sold (Cr)'], errors='coerce')
df_final = df_final.dropna(subset=['Sold (Cr)'])
print(df_final['Sold (Cr)'].unique())
print("\nSuccess! Column Type:", df_final['Sold (Cr)'].dtype)


In [0]:
df_final

In [0]:
df_final.drop(["join_key","team_bat","team_bowl","Target_Auction_Year"], axis=1, inplace=True)

In [0]:
df_final.sort_values(by='Sold (Cr)', ascending=False, inplace=True)
df_final

In [0]:
df_final[(df_final['Is_Debutant'] == 1) & (df_final['Sold (Cr)'] > 0)]

In [0]:
# Check examples of both debutants and non-debutants
print("=== DEBUTANTS (Is_Debutant = 1) ===")
debutants = df_final[df_final['Is_Debutant'] == 1][['player', 'matches_bat', 'matches_bowl', 'Is_Debutant']].head(10)
print(debutants)

print("\n=== NON-DEBUTANTS (Is_Debutant = 0) ===")
non_debutants = df_final[df_final['Is_Debutant'] == 0][['player', 'matches_bat', 'matches_bowl', 'Is_Debutant']].head(10)
print(non_debutants)

print(f"\n=== SUMMARY ===")
print(f"Total players: {len(df_final)}")
print(f"Debutants: {df_final['Is_Debutant'].sum()}")
print(f"Non-debutants: {(df_final['Is_Debutant'] == 0).sum()}")

# Check if there are any non-debutants with 0 matches in both
print("\n=== SUSPICIOUS: Non-debutants with 0 matches in both ===")
suspicious = df_final[(df_final['Is_Debutant'] == 0) & (df_final['matches_bat'] == 0) & (df_final['matches_bowl'] == 0)][['player', 'matches_bat', 'matches_bowl', 'Is_Debutant', 'Auction Year']]
print(f"Count: {len(suspicious)}")
print(suspicious.head(10))

In [0]:
output_path = "/Workspace/Users/manikanthgoud98@gmail.com/ipl_data_modelling/data/processed"
os.makedirs(output_path, exist_ok=True)
df_final.to_csv(os.path.join(output_path, "ipl_data.csv"))